## Concise, barebones workflow for getting to a TSM Line

#### Imports

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import pandas as pd
import numpy as np
import scipy as sci
import matplotlib.pyplot as pl
import matplotlib.image as img
import subprocess
import pathlib
import pyvista as pv
import copy

import fenics_sz.utils
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry
from fenics_sz.sz_problems.sz_steady_dislcreep import SteadyDislSubductionProblem
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem
from fenics_sz.sz_problems.sz_params import default_params, allsz_params

In [ ]:
from fenics_sz.fluid_release.perple_x_integration import get_PT_data_from_tabs, plot_PT_data
import fenics_sz.fluid_release.get_PT_curves 

#### Create and Solve SZ

In [ ]:
resscale = 5.0
n_honshu_dict = allsz_params["48_N_Honshu"]
n_honshu_slab = create_slab(n_honshu_dict['xs'], n_honshu_dict['ys'], resscale, n_honshu_dict['lc_depth'])
plot_slab(n_honshu_slab)

In [ ]:
n_honshu_dict

In [ ]:
n_honshu_dict['xs'][-1]

In [ ]:
n_honshu_geom = create_sz_geometry(n_honshu_slab, resscale, n_honshu_dict['sztype'], n_honshu_dict['io_depth'], n_honshu_dict['extra_width'], 
                             n_honshu_dict['coast_distance'], n_honshu_dict['lc_depth'], n_honshu_dict['uc_depth'])
n_honshu_sz = TDDislSubductionProblem(n_honshu_geom, **n_honshu_dict)

n_honshu_sz.solve(n_honshu_dict['As'], dt=0.05, theta=0.5, rtol=1.e-1, verbosity=1)

plotter = pv.Plotter()
fenics_sz.utils.plot.plot_scalar(n_honshu_sz.T_i, plotter=plotter, scale=n_honshu_sz.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
# fenics_sz.utils.plot.plot_vector_glyphs(n_honshu_sz.vw_i, plotter=plotter, gather=True, factor=0.1, color='k', scale=fenics_sz.utils.mps_to_mmpyr(n_honshu_sz.v0))
# fenics_sz.utils.plot.plot_vector_glyphs(n_honshu_sz.vs_i, plotter=plotter, gather=True, factor=0.1, color='k', scale=fenics_sz.utils.mps_to_mmpyr(n_honshu_sz.v0))
n_honshu_geom.pyvistaplot(plotter=plotter, color='green', width=2)
cdpt = n_honshu_slab.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter, render_points_as_spheres=True, point_size=10.0, color='green')
fenics_sz.utils.plot.plot_show(plotter)
# fenics_sz.utils.plot.plot_save(plotter, output_folder / "{}_td_solution_resscale_{:.2f}.png".format("kamchatka", resscale))

## Workflow for producing a TSM plot

* generate P-T-H2O% plots for the minerologies in the slab

* get points along slab in a slab-tangent slab-normal coordinate system
* probe the depths and temperatures at those points
* convert the depths to pressures at each point
* use the pressure-temperature data at each point to get a hydration % for each point
* (use groups of four points to get an averaged H2O % for the cell the points create?)
* calculate the area of each cell
* convert cell area to weight per meter of trench
* convert weight per meter of trench to water weight
* wherever there is a change in water content along a slab normal path, catalog the average depth of the two cells across which the change occurs.


### Test to make sure a slab orthogonal grid can be made

In [ ]:
def in_domain(sz, point):
    return ((0 < point[0] < int(sz.geom.slab_spline(1)[0])) and (int(sz.geom.slab_spline(1)[1]) < point[1] < 0))

In [ ]:
# check where things will be out of bounds before making the big list of points:

def get_regular_grid_attempt_3(sz, h_serp, u_res, depth_res):
    xys = []
    spline_dist = []
    us = np.linspace(0,1,u_res)

    test_spline = copy.deepcopy(sz.geom.slab_spline)
    test_spline.translatenormal((-7 -h_serp))
    test_xys = np.asarray([test_spline(u) + [0.0] for u in us])
    good_us = [] # list of u values to use when creating the set of xy points in slab space 
    for u in range(u_res):
        if in_domain(sz, test_xys[u]) ==True:
            good_us.append(us[u])
    print(good_us)


    for i in range(depth_res):
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(i * ((-7 -h_serp)/depth_res))
        xys.append(np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))

    xys_as_array = np.asarray(xys)
    return xys_as_array, spline_dist

In [ ]:
# manually defined layer depths for everything above the mantle:

def get_regular_grid_attempt_4(sz, h_serp, u_res):
    xys = []
    spline_dist = []
    spline_depth = []
    us = np.linspace(0,1,u_res)

    test_spline = copy.deepcopy(sz.geom.slab_spline)
    test_spline.translatenormal((-7 -h_serp))
    test_xys = np.asarray([test_spline(u) + [0.0] for u in us])
    good_us = [] # list of u values to use when creating the set of xy points in slab space 
    for u in range(u_res):
        if in_domain(sz, test_xys[u]) ==True:
            good_us.append(us[u])
    print(good_us)

#upper and lower volcanics
    depth = 0
    volcanics_top = copy.deepcopy(sz.geom.slab_spline)
    xys.append(np.asarray([volcanics_top(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_top.length for u in good_us]))
    spline_depth.append(depth)

    depth = -0.3
    volcanics_middle = copy.deepcopy(sz.geom.slab_spline)
    volcanics_middle.translatenormal(depth)
    xys.append(np.asarray([volcanics_middle(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_middle.length for u in good_us]))
    spline_depth.append(depth)

    depth = -0.6
    volcanics_bottom = copy.deepcopy(sz.geom.slab_spline)
    volcanics_bottom.translatenormal(depth)
    xys.append(np.asarray([volcanics_bottom(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_bottom.length for u in good_us]))
    spline_depth.append(depth)


# dikes
    depth = -1.3
    dikes_middle = copy.deepcopy(sz.geom.slab_spline)
    dikes_middle.translatenormal(depth)
    xys.append(np.asarray([dikes_middle(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*dikes_middle.length for u in good_us]))
    spline_depth.append(depth)

    depth = -2.0
    dikes_bottom = copy.deepcopy(sz.geom.slab_spline)
    dikes_bottom.translatenormal(depth)
    xys.append(np.asarray([dikes_bottom(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*dikes_bottom.length for u in good_us]))
    spline_depth.append(depth)

# gabbros:
    for i in range(5): #1km spaced layers in gabbros (5km thick)
        depth = (-2.0 - (i+1)) # start of mantle depth plus depth of increments w/n mantle
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(depth)
        xys.append(np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))
        spline_depth.append(depth)


# mantle:
    for i in range(h_serp): # 1km spaced lines in the serp. mantle
        depth = (-7.0 - (i+1)) # start of mantle depth plus depth of increments w/n mantle
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(depth)
        xys.append(np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))
        spline_depth.append(depth)

    xys_as_array = np.asarray(xys)
    return xys_as_array, spline_dist, spline_depth

In [ ]:
dist_res = 100

In [ ]:
h_serp = 2

In [ ]:
regular_points, spline_dist, layer_depths = get_regular_grid_attempt_4(n_honshu_sz, h_serp, dist_res)

In [ ]:
# print(len(layer_depths))

for i in range(len(layer_depths)):
    print(layer_depths[i])

In [ ]:
print(len(regular_points))
print(len(regular_points[0])) #thinner h_serp gives you more valid slab dists. to wok with
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print((regular_points[11][43]))


In [ ]:
print(len(spline_dist))
print(len(spline_dist[0]))

for i in range(len(spline_dist)):
    print(spline_dist[i][94])

#### Check to make sure points fall in a grid

In [ ]:
plotter = pv.Plotter()
fenics_sz.utils.plot.plot_scalar(n_honshu_sz.T_i, plotter=plotter, scale=n_honshu_sz.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
n_honshu_geom.pyvistaplot(plotter=plotter, color='green', width=2)
cdpt = n_honshu_slab.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter, render_points_as_spheres=True, point_size=10.0, color='green')

for i in range(len(regular_points)):
    for j in range(len(regular_points[i])):            
        fenics_sz.utils.plot.plot_points([[regular_points[i][j][0], regular_points[i][j][1], 0.0]], plotter=plotter, point_size=0.5, color='black')

fenics_sz.utils.plot.plot_show(plotter)

#### Precalculating temperatures

In [ ]:
print(len(regular_points))
print(len(regular_points[0]))
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print(type(regular_points[0][0]))


In [ ]:
def get_interpolator(perple_x_data):
    PT_points = (perple_x_data[0], perple_x_data[1])
    predict = sci.interpolate.RegularGridInterpolator(PT_points, perple_x_data[2], method = 'linear')
    return predict

    #create an interpolator object given the p, t, h2o perplex output

def predict_h2o(interpolator, t, p):
    # print("Temp: ", t)
    # print("Pressure: ", p)
    if t < 200:
        return interpolator([p, 200])
    # case for if the temperatures fall outside the bounds of the data used to generate the interpolator

    else:
        return (interpolator([p, t]))

    #predict the %wt hydration at a point given the pressure and temperature

In [ ]:
DMM_data = get_PT_data_from_tabs('DMMdamp_25')
uvolcs_data = get_PT_data_from_tabs('upvolc_25')
lvolcs_data = get_PT_data_from_tabs('lovolc_25')
dikes_data = get_PT_data_from_tabs('dike_25')
gabbros_data = get_PT_data_from_tabs('gabbro_25')

In [ ]:
uvolcs_interp = get_interpolator(uvolcs_data)
lvolcs_interp = get_interpolator(lvolcs_data)
dikes_interp = get_interpolator(dikes_data)
gabbros_interp = get_interpolator(gabbros_data)
damp_DMM_interp = get_interpolator(DMM_data)

In [ ]:
ts = []
zs = []
ps = []

for i in range(len(regular_points)):
    cinds, cells = fenics_sz.utils.mesh.get_cell_collisions(regular_points[i], n_honshu_sz.mesh)
    t, z, p = (n_honshu_sz.T_i.eval(regular_points[i], cells)[:,0], -regular_points[i][:,1], (-regular_points[i][:,1])*1000 * 3300 * 9.81 *1e-9)
    ts.append(t)
    zs.append(z)
    ps.append(p)

# ts, zs, ps, are all 2d arrays. first index gives the layer, second index gives the u.
# value is just value of the point. 

In [ ]:
print(len(zs))
print(len(zs[0]))

for i in range(len(zs)):
    print(zs[i][94])

## The Cell class

In [ ]:
# pass in a PT H2O interpolator when initializing the class, instead of the perpleX data
# one interpolator per lithology

# i can also try structuring it such that the temp of each vertex is passed in as a parameter.
# this way, mesh-point collisions and temperature evals are only done once per point, as opposed to four times


class Cell:
    def __init__(self, sz, slab_depth_1, slab_depth_2, slab_dist_1, slab_dist_2,
                 z1, z2, z3, z4, p1, p2, p3, p4, t1, t2, t3, t4, interpolator):
        self.sz = sz
        self.slab_depth_1 = slab_depth_1
        self.slab_depth_2 = slab_depth_2
        self.slab_dist_1 = slab_dist_1
        self.slab_dist_2 = slab_dist_2
        self.pressures = np.array([p1, p2, p3, p4])
        self.temps = np.array([t1, t2, t3, t4])
        self.vertex_depths = np.array([z1, z2, z3, z4])
        self.interpolator = interpolator
        self._water_pct = None
        self._hydrations = []
    
    def get_area(self):
        return((self.slab_depth_2 - self.slab_depth_1) * (self.slab_dist_2 - self.slab_dist_1))

    def get_depth(self):
        return(sum(self.vertex_depths)/4)

    def get_high_depth(self):
        pass

    def get_low_depth(self):
        pass

    def get_temp(self):
        return(sum(self.temps)/4)

    def get_water_pct(self):
        if self._water_pct is None:

            for i in range(len(self.temps)):
                if self.temps[i] < 200:
                    self._hydrations.append(self.interpolator([self.pressures[i], 200]))
                # case for if the temperatures fall outside the bounds of the data used to generate the interpolator

                else:
                    self._hydrations.append(self.interpolator([self.pressures[i], self.temps[i]]))

            self._water_pct = sum(self._hydrations)/4
        return self._water_pct

    def get_water_wt(self):
        return (self.get_water_pct() * 1e-2 * self.get_area() * 3300 * 1e-3)
        # units documentation:
        # first putting percents in decimal form
        # multiplying by area to get km^2

        # multiply density by 1e9 to get the density in units of kg/km^3
        # multiply current value by 1e-9 to convert from kg to Tg
        # two conversions above cancel out

        # left with units of Tg/km
        # multiply by 1e-3 to get Tg/m



    def __repr__(self):
        pass


### Initializing cells by indexing over regular points

In [ ]:
print(len(regular_points))
print(len(regular_points[0]))
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print(type(regular_points[0][0]))

In [ ]:
n_honshu_cells = []
interp = None
for i in range(len(layer_depths)-1):
    layer_cells = []

    if layer_depths[i+1] >= -0.3:
        interp = uvolcs_interp
    elif layer_depths[i+1] >= -0.6:
        interp = lvolcs_interp
    elif layer_depths[i+1] >= -2:
        interp = dikes_interp
    elif layer_depths[i+1] >= -7:
        interp = gabbros_interp
    elif layer_depths[i+1] >= -7 - h_serp:
        interp = damp_DMM_interp
    else:
        raise Exception("Improper cell depth")


    for j in range(len(regular_points[i])-1): # problem with area approx is that spline dist varies between layers (taking average of spline dist btwn cell's 2 defining layers might make it a bit better)
        cell = Cell(n_honshu_sz, layer_depths[i], layer_depths[i+1], spline_dist[i][j], spline_dist[i][j+1],
                    zs[i][j], zs[i][j+1], zs[i+1][j], zs[i+1][j+1], ps[i][j], ps[i][j+1], ps[i+1][j], ps[i+1][j+1], 
                    ts[i][j], ts[i][j+1], ts[i+1][j], ts[i+1][j+1], interp)
        # might want to consider going back to passing in an interpolator object, and doing the hydration prediction within the class
        layer_cells.append(cell)
    n_honshu_cells.append(layer_cells)

#### some code to check what interpolators are actually being passed into the cells:

In [ ]:

interp = None
for i in range(len(regular_points)-1):
    if layer_depths[i+1] >= -0.3:
        interp = "uvolcs_interp"
    elif layer_depths[i+1] >= -0.6:
        interp = "lvolcs_interp"
    elif layer_depths[i+1] >= -2:
        interp = "dikes_interp"
    elif layer_depths[i+1] >= -7:
        interp = "gabbros_interp"
    elif layer_depths[i+1] >= -7 - h_serp:
        interp = "damp_DMM_interp"
    else:
        raise Exception("Improper cell depth")
    
    print("Depth of Cell Bottom: " , layer_depths[i+1], " -------- ",  "Interpolator: " , interp)

    for j in range(len(regular_points[i])-1): # problem with area approx is that spline dist varies between layers (taking average of spline dist btwn cell's 2 defining layers might make it a bit better)
        pass

#currently resolution is too coarse to pick up the either of the volcanics layer
# need a resolutoin that gives a step size of at most 0.3 in order to pick that up

# with h_serp = 2 it picks up the volcanics!

#### Getting cell hydrations:

In [ ]:
cell_hydrations = []
for i in range(len(n_honshu_cells)):
    layer_cell_hydrations = []
    for j in range(len(n_honshu_cells[i])):
        layer_cell_hydrations.append(n_honshu_cells[i][j].get_water_pct()[0])
    cell_hydrations.append(layer_cell_hydrations)

In [ ]:
print(len(cell_hydrations))
print(len(cell_hydrations[0]))
print((cell_hydrations[0][0]))

In [ ]:
print(len(n_honshu_cells))
print(len(n_honshu_cells[0]))
# print(len(n_honshu_cells[0][0]))

In [ ]:
fig, ax = pl.subplots()

slab_surf_dist = [] # distance along slab *surface* , not along the layer
for i in range(len(spline_dist[0])):
    slab_surf_dist.append(spline_dist[0][i])


c = ax.pcolor(slab_surf_dist, layer_depths, cell_hydrations, cmap='Blues')
ax.set_title('n_honshu Cell HYDRATIONS; 5x Slab-Normal Exageration')
ax.set_xlabel('Distance along slab surface (km)')
ax.set_ylabel('Distance normal to slab surface (km)')

slab_normal_exag = 5
ax.set_box_aspect((-layer_depths[-1] / slab_surf_dist[-1]) * slab_normal_exag)


fig.colorbar(c, ax=ax)
pl.show()

#### Cell Temperatures

In [ ]:
# this prints temperatures along a slab-normal line!
for i in range(len(n_honshu_cells)):
    print(n_honshu_cells[i][80].get_temp())

cell_temps = []
for i in range(len(n_honshu_cells)):
    layer_cell_temps = []
    for j in range(len(n_honshu_cells[i])):
        layer_cell_temps.append(n_honshu_cells[i][j].get_temp())
    cell_temps.append(layer_cell_temps)

In [ ]:
fig, ax = pl.subplots()

slab_surf_dist = [] # distance along slab *surface* , not along the layer
for i in range(len(spline_dist[0])):
    slab_surf_dist.append(spline_dist[0][i])


c = ax.pcolor(slab_surf_dist, layer_depths, cell_temps, cmap='RdBu_r')
ax.set_title('n_honshu Cell Temps; 5x Slab-Normal Exageration')
ax.set_xlabel('Distance along slab surface (km)')
ax.set_ylabel('Distance normal to slab surface (km)')

slab_normal_exag = 5
ax.set_box_aspect((-layer_depths[-1] / slab_surf_dist[-1]) * slab_normal_exag)


fig.colorbar(c, ax=ax)
pl.show()


#### Disallowing rehydration

In [ ]:
def remove_rehydration(cell_hydrations):
    for i in range(len(cell_hydrations)):
        for j in range(len(cell_hydrations[i])-1):
            if cell_hydrations[i][j+1] > cell_hydrations[i][j]:
                cell_hydrations[i][j+1] = cell_hydrations[i][j]
    # FIXME more of a warning: this function transforms the input, it doesn't create a new variable
    # that is, when used, an initial hydrations array that allows for rehydration is not retained
    return cell_hydrations

In [ ]:
n_honshu_cell_hydrations_no_rehydration = remove_rehydration(cell_hydrations)

In [ ]:
fig, ax = pl.subplots()

slab_surf_dist = [] # distance along slab *surface* , not along the layer
for i in range(len(spline_dist[0])):
    slab_surf_dist.append(spline_dist[0][i])


c = ax.pcolor(slab_surf_dist, layer_depths, cell_hydrations, cmap='Blues') 
# this should actually be plotting hydrations considering allowing for rehydration...
ax.set_title('n_honshu Cell Hydration NO REHYDRATION; 5x Slab-Normal Exageration')
ax.set_xlabel('Distance along slab surface (km)')
ax.set_ylabel('Distance normal to slab surface (km)')

slab_normal_exag = 5
ax.set_box_aspect((-layer_depths[-1] / slab_surf_dist[-1]) * slab_normal_exag)


fig.colorbar(c, ax=ax)
pl.show()

#### Water Loss

In [ ]:
def get_water_loss(cells, cell_hydrations):
    slab_losses_and_depths = []
    for i in range(len(cell_hydrations)):
        for j in range(len(cell_hydrations[i])-1):
            if (cell_hydrations[i][j+1] < cell_hydrations[i][j]):
                # should I check against an epsilon instead?

                # losses_and_depths = [cell_hydrations[i][j] - cell_hydrations[i][j+1] , ((cells[i][j].get_depth() + cells[i][j+1].get_depth()) / 2)]


                #This line stores total water lost, not water pct. water pct needs to be used for comparison, not water loss (I think...)
                losses_and_depths = [cells[i][j+1].get_water_wt() - cells[i][j].get_water_wt() , ((cells[i][j].get_depth() + cells[i][j+1].get_depth()) / 2)]
                
                #FIXME depths are currently global depths, not depths to surface of the slab

                # water_losses.append(cell_hydrations[i][j] - cell_hydrations[i][j+1])
                # water_loss_depths.append(((cells[i][j].get_depth() + cells[i][j+1].get_depth()) / 2))
                slab_losses_and_depths.append(losses_and_depths)

    return slab_losses_and_depths


In [ ]:
n_honshu_water_losses_and_depths = get_water_loss(n_honshu_cells, n_honshu_cell_hydrations_no_rehydration)

In [ ]:
print(len(n_honshu_water_losses_and_depths))
print(len(n_honshu_water_losses_and_depths[0]))

# for i in range(len(n_honshu_water_losses_and_depths)):
#     print(n_honshu_water_losses_and_depths[i][1])

sorted_n_honshu_water_losses_and_depths = sorted(n_honshu_water_losses_and_depths, key=lambda l:l[1])
print("list sorted")


print(len(n_honshu_water_losses_and_depths))
print(len(n_honshu_water_losses_and_depths[0]))


# for i in range(len(sorted_n_honshu_water_losses_and_depths)):
#     print(sorted_n_honshu_water_losses_and_depths[i][1]) #check that depths are actually sorted



for i in range(len(sorted_n_honshu_water_losses_and_depths)):
    print(sorted_n_honshu_water_losses_and_depths[i][0]) #check what the water losses are (should be in Tg/m)


In [ ]:
# I think my code is flagging too many cells as having water loss
# plotting water loss as a function of depth to see if most of themflagged cells have minimal, next to no water loss

fig, ax = pl.subplots()
ax.plot([row[0] for row in sorted_n_honshu_water_losses_and_depths], [row[1] for row in sorted_n_honshu_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('n_honshu water loss (delta (Tg/m)) as a function of depth (km)')
ax.set_xlabel('Tg/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')



pl.show()

### Converting to Tokyo subway map units: Tg/MYr/m

#### Cumulative sum

In [ ]:
# performing cumulative sum on Tg/m loss array to check if shape aligns w literature

cum_sum_array = []
cum_sum = 0
for i in range(len(sorted_n_honshu_water_losses_and_depths)):
    cum_sum += sorted_n_honshu_water_losses_and_depths[i][0]
    print(cum_sum)
    cum_sum_array.append(cum_sum[0])
    print(cum_sum_array)

In [ ]:
fig, ax = pl.subplots()
ax.plot(cum_sum_array, [row[1] for row in sorted_n_honshu_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('n_honshu CUMULATIVE water loss (delta (Tg/m)) as a function of depth (km)')
ax.set_xlabel('Tg/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')



pl.show()

#### Conversion to per unit time

In [ ]:
print(n_honshu_dict['Vs'])

In [ ]:
n_honshu_distance_increment = n_honshu_sz.geom.slab_spline.length / dist_res
# (in km) slab-tangent distance between two st points on the surface of the n_honshu slab

print(n_honshu_distance_increment)

In [ ]:
time_standarized_losses = []

for i in range(len(sorted_n_honshu_water_losses_and_depths)):
    time_standarized_losses.append(sorted_n_honshu_water_losses_and_depths[i][0][0] * n_honshu_dict['Vs'] / n_honshu_distance_increment)

In [ ]:
print(len(time_standarized_losses))

In [ ]:
fig, ax = pl.subplots()
ax.plot(time_standarized_losses, [row[1] for row in sorted_n_honshu_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('n_honshu time-standardized water loss (delta (Tg/MYr/m)) as a function of depth (km)')
ax.set_xlabel('Tg/MYr/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')



pl.show()

In [ ]:
cum_sum_time_standardized_array = []
cum_sum_time_standardized = 0
for i in range(len(time_standarized_losses)):
    cum_sum_time_standardized += time_standarized_losses[i]
    print(cum_sum_time_standardized)
    cum_sum_time_standardized_array.append(cum_sum_time_standardized)
    print(cum_sum_time_standardized_array)


In [ ]:
fig, ax = pl.subplots()
ax.plot(cum_sum_time_standardized_array, [row[1] for row in sorted_n_honshu_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('n_honshu CUMULATIVE TIME-STANDARDIZED water loss (delta (Tg/MYr/m)) as a function of depth (km)')
ax.set_xlabel('Tg/MYr/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')

ax.set_box_aspect(2.5)


pl.show()

print("Total water loss: " , cum_sum_time_standardized)

Considering I'm excluding sediments, I'm only 1-2 Tg/MYr/m off of Geoff's shear heating paper

I do still need to include a sediment layer, and figure out a way to make depths be depths to the surface of the slab (not depth of the cell itself). Adding sediments will further increase my water loss.

Things that have come up: not sure if slab noral lines are actually normal